# Teoría de Random Forest

## 1. Introducción

**Random Forest** (Bosque Aleatorio) es un algoritmo de **aprendizaje ensamblado (ensemble learning)** que construye múltiples árboles de decisión durante el entrenamiento y combina sus predicciones para obtener un resultado más preciso y robusto.

### Concepto Clave: Ensemble Learning

**Ensemble Learning** combina múltiples modelos débiles (weak learners) para crear un modelo fuerte (strong learner):

$$
\text{Predicción Final} = \text{Agregación}(\text{Modelo}_1, \text{Modelo}_2, ..., \text{Modelo}_n)
$$

**Ventajas del ensemble:**
* Reduce **overfitting** (sobreajuste)
* Mejora **accuracy** (precisión)
* Mayor **robustez** ante datos ruidosos
* Reduce **varianza** del modelo

### Historia

* **1995**: Leo Breiman introduce **Bagging** (Bootstrap Aggregating)
* **2001**: Leo Breiman publica el paper "Random Forests"
* **Hoy**: Uno de los algoritmos más usados en competencias de ML (Kaggle)

---

## 2. Algoritmo Random Forest

### Pseudocódigo

```
Entrada: Dataset D = {(x₁, y₁), (x₂, y₂), ..., (xₙ, yₙ)}
         Número de árboles: T
         Número de features por árbol: m

Para t = 1 hasta T:
    1. Bootstrap: Crear muestra Dₜ de D con reemplazo (mismo tamaño)
    2. Feature Sampling: Seleccionar aleatoriamente m features
    3. Entrenar árbol de decisión Tₜ con Dₜ y m features
    4. Sin poda (árboles crecen completamente)

Predicción:
    - Clasificación: Voto mayoritario de los T árboles
    - Regresión: Promedio de las predicciones de los T árboles
```

### Proceso Visual

```
Dataset Original
      ↓
   Bootstrap
   /    |    \
  D₁   D₂   D₃  ...  Dₜ  (T muestras con reemplazo)
  ↓    ↓    ↓         ↓
Tree₁ Tree₂ Tree₃ ... Treeₜ (T árboles)
  ↓    ↓    ↓         ↓
  P₁   P₂   P₃  ...  Pₜ  (T predicciones)
   \    |    /         /
    \   |   /        /
     Agregación (Voto/Promedio)
           ↓
   Predicción Final
```

### Componentes Clave

#### 1. **Bagging (Bootstrap Aggregating)**

Cada árbol se entrena con una muestra **bootstrap** (con reemplazo):

$$
D_t = \{\text{sample}(D, n, \text{replace=True})\}
$$

* Cada $D_t$ tiene el mismo tamaño que $D$
* Aproximadamente **63%** de las muestras aparecen en cada bootstrap
* **37%** quedan fuera (**Out-Of-Bag samples** o OOB)

#### 2. **Feature Randomness**

En cada división de nodo, solo se consideran **m features aleatorias**:

* **Clasificación**: $m = \sqrt{p}$ (raíz cuadrada del total de features)
* **Regresión**: $m = p/3$ (un tercio del total de features)
* Donde $p$ es el número total de features

Esto **decorrelaciona** los árboles, reduciendo la varianza.

#### 3. **Agregación de Predicciones**

**Para Clasificación (voto mayoritario):**

$$
\hat{y} = \text{mode}(T_1(x), T_2(x), ..., T_n(x))
$$

**Para Regresión (promedio):**

$$
\hat{y} = \frac{1}{T} \sum_{t=1}^{T} T_t(x)
$$

---

## 3. Ventajas y Desventajas

### ✅ Ventajas

1. **Alta Precisión**: Supera a un solo árbol de decisión
2. **Robustez**:
   * Maneja bien **outliers** y datos ruidosos
   * No requiere normalización de features
   * Funciona con features categóricas y numéricas
3. **Prevención de Overfitting**: La aleatoriedad reduce sobreajuste
4. **Feature Importance**: Mide la importancia de cada variable
5. **OOB Error**: Estimación de error sin necesidad de validación cruzada
6. **Paralelizable**: Los árboles se entrenan independientemente
7. **Versatilidad**: Funciona para clasificación y regresión

### ❌ Desventajas

1. **Menos Interpretable**: Caja negra (vs un solo árbol)
2. **Computacionalmente Costoso**: Entrena T árboles
3. **Predicción Lenta**: Debe evaluar T árboles (vs 1 árbol)
4. **Tamaño del Modelo**: Ocupa más memoria (T árboles en RAM)
5. **Extrapolación Pobre**: No predice bien fuera del rango de entrenamiento
6. **Sesgado hacia Features con Muchas Categorías**: En clasificación

### Comparación: Random Forest vs Decision Tree

| Aspecto | Decision Tree | Random Forest |
|---------|---------------|---------------|
| **Accuracy** | Menor | Mayor |
| **Overfitting** | Alto riesgo | Bajo riesgo |
| **Interpretabilidad** | Alta | Baja |
| **Velocidad entrenamiento** | Rápido | Lento |
| **Velocidad predicción** | Muy rápido | Moderado |
| **Robustez** | Baja | Alta |
| **Feature Importance** | Sí | Sí (más confiable) |

---

## 4. Hiperparámetros Clave

### Parámetros de Ensemble

1. **`numTrees` (número de árboles)**
   * Más árboles → Mayor accuracy, pero más lento
   * **Típico**: 100-500 árboles
   * **Regla**: Aumentar hasta que el error se estabilice

2. **`featureSubsetStrategy` (m features por split)**
   * **Clasificación**: `"sqrt"` ($m = \sqrt{p}$)
   * **Regresión**: `"onethird"` ($m = p/3$)
   * **Otros**: `"log2"`, `"all"`, o número fijo

### Parámetros de Árboles Individuales

3. **`maxDepth` (profundidad máxima)**
   * Random Forest suele usar árboles profundos (sin poda)
   * **Típico**: 10-30 (o sin límite)

4. **`minInstancesPerNode` (mínimo de ejemplos por hoja)**
   * Controla el tamaño mínimo de las hojas
   * **Típico**: 1-5

5. **`maxBins` (número de bins para discretización)**
   * Mayor → Más splits posibles, pero más lento
   * **Típico**: 32

### Parámetros de Bootstrap

6. **`subsamplingRate` (fracción de datos por árbol)**
   * **Por defecto**: 1.0 (bootstrap con reemplazo)
   * **Alternativa**: 0.8 (80% sin reemplazo, más rápido)

### Regla de Oro para Hiperparámetros

```python
# Configuración recomendada inicial
rf = RandomForestClassifier(
    numTrees=100,                    # Aumentar si tienes recursos
    featureSubsetStrategy="sqrt",    # sqrt para clasificación
    maxDepth=None,                   # Sin límite (árboles profundos)
    minInstancesPerNode=1,           # Hojas pequeñas OK
    subsamplingRate=1.0              # Bootstrap completo
)
```

---

## 5. Feature Importance

Random Forest calcula la **importancia de cada feature** basándose en cuánto reducen la impureza (Gini o entropía) en promedio.

### Cálculo

Para cada feature $f$:

1. En cada árbol $t$, sumar la reducción de impureza de todos los splits que usan $f$:
   $$
   \text{Importance}_t(f) = \sum_{\text{nodos que usan } f} \Delta \text{Impureza}
   $$

2. Promediar sobre todos los árboles:
   $$
   \text{Importance}(f) = \frac{1}{T} \sum_{t=1}^{T} \text{Importance}_t(f)
   $$

3. Normalizar para que sumen 1:
   $$
   \text{Importance}(f) = \frac{\text{Importance}(f)}{\sum_{f'} \text{Importance}(f')}
   $$

### Interpretación

* **Valores altos**: Feature muy importante para las predicciones
* **Valores bajos**: Feature poco relevante (candidata a eliminar)
* **Suma = 1.0**: Las importancias son proporciones

### Ejemplo

```
Feature Importance:
1. Monthly_Charges    : 0.35  (35% de importancia)
2. Tenure              : 0.28
3. Contract_Type       : 0.15
4. Total_Charges       : 0.12
5. Internet_Service    : 0.07
6. Payment_Method      : 0.03
```

**Interpretación**: `Monthly_Charges` es el factor más importante para predecir churn.

### Uso Práctico

* **Feature Selection**: Eliminar features con importancia < 0.01
* **Interpretabilidad**: Explicar qué variables importan más
* **Domain Insights**: Validar hipótesis de negocio

---

## 6. Out-Of-Bag (OOB) Error

### Concepto

En cada bootstrap, aproximadamente **37%** de las muestras quedan **fuera** (Out-Of-Bag).

Estas muestras OOB se pueden usar como **conjunto de validación implícito**:

1. Para cada muestra $x_i$, usar solo los árboles que **NO** la incluyeron en su bootstrap
2. Predecir $x_i$ con esos árboles
3. Comparar con $y_i$ para calcular el error

### Ventajas

* **No requiere train/test split**: OOB simula un conjunto de validación
* **Uso eficiente de datos**: Todo el dataset se usa para entrenar
* **Estimación no sesgada**: Similar a cross-validation

### Fórmula

**OOB Error (clasificación):**

$$
\text{OOB Error} = \frac{1}{n} \sum_{i=1}^{n} I(y_i \neq \hat{y}_i^{\text{OOB}})
$$

**OOB Error (regresión):**

$$
\text{OOB Error} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i^{\text{OOB}})^2
$$

Donde $\hat{y}_i^{\text{OOB}}$ es la predicción usando solo árboles que no vieron $x_i$.

### Uso Práctico

```python
# En PySpark ML, obtener OOB error:
rf_model = rf.fit(train_data)
oob_error = rf_model.oobError  # Si está habilitado
print(f"OOB Error: {oob_error:.4f}")
```

**Interpretación**: Si OOB Error ≈ Test Error, el modelo generaliza bien.

---

## 7. Conclusiones

### Cuándo Usar Random Forest

✅ **Úsalo cuando:**
* Necesitas alta precisión
* Tienes suficiente tiempo de entrenamiento
* No requieres interpretabilidad extrema
* Tienes features mixtas (numéricas y categóricas)
* Datos ruidosos o con outliers

❌ **Evítalo cuando:**
* Necesitas interpretabilidad perfecta (usa Decision Tree)
* Tiempo de predicción es crítico (usa modelos lineales)
* Dataset muy pequeño (< 1000 muestras)
* Recursos computacionales limitados
* Necesitas extrapolación fuera del rango de entrenamiento

### Comparación con Otros Algoritmos

| Algoritmo | Accuracy | Velocidad | Interpretabilidad | Robustez |
|-----------|----------|-----------|-------------------|----------|
| **Decision Tree** | ⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐ |
| **Random Forest** | ⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐ | ⭐⭐⭐⭐⭐ |
| **Gradient Boosting** | ⭐⭐⭐⭐⭐ | ⭐⭐ | ⭐⭐ | ⭐⭐⭐⭐ |
| **Logistic Regression** | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐ |

### Fórmulas Clave para Recordar

1. **Features por split (clasificación)**: $m = \sqrt{p}$
2. **Features por split (regresión)**: $m = p/3$
3. **Predicción (clasificación)**: $\hat{y} = \text{mode}(T_1(x), ..., T_n(x))$
4. **Predicción (regresión)**: $\hat{y} = \frac{1}{T} \sum_{t=1}^{T} T_t(x)$
5. **OOB muestras por árbol**: ~37% del dataset

### Próximos Pasos

1. **Práctica**: Implementar Random Forest en PySpark ML
2. **Comparar**: Random Forest vs Decision Tree en el mismo problema
3. **Optimizar**: Tunear hiperparámetros con Grid Search
4. **Avanzado**: Explorar Gradient Boosted Trees (GBT)

---

## 📚 Referencias

* **Paper Original**: Breiman, L. (2001). "Random Forests". Machine Learning, 45(1), 5-32.
* **PySpark ML**: [Random Forest Documentation](https://spark.apache.org/docs/latest/ml-classification-regression.html#random-forest-classifier)
* **Scikit-Learn**: [Random Forest Guide](https://scikit-learn.org/stable/modules/ensemble.html#forest)

---

**¡Ahora estás listo para aplicar Random Forest en problemas reales!** 🌲🌲🌲